In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('WELFake_Dataset.csv')

In [3]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
df.shape

(72134, 4)

In [5]:
df.columns

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='str')

In [6]:
df.dtypes

Unnamed: 0    int64
title           str
text            str
label         int64
dtype: object

In [7]:
df['Fake News']=df['title']+' '+df['text']

In [8]:
df=df.drop(columns=['Unnamed: 0','title','text'])

In [9]:
df.head()

,label,Fake News
0,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,NaN
2,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,1,SATAN 2: Russia unvelis an image of its terrif...


In [10]:
df.isnull().sum()

label          0
Fake News    597
dtype: int64

In [11]:
df=df.dropna()

In [12]:
df.isnull().sum()

label        0
Fake News    0
dtype: int64

In [13]:
df.shape

(71537, 2)

In [14]:
df.duplicated().sum()

np.int64(8416)

In [15]:
df[df.duplicated(keep=False)]

,label,Fake News
4,1,SATAN 2: Russia unvelis an image of its terrif...
5,1,About Time! Christian Group Sues Amazon and SP...
7,1,HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...
8,1,Sports Bar Owner Bans NFL Games…Will Show Only...
13,1,WATCH: HILARIOUS AD Calls Into Question Health...
...,...,...
72116,1,SAY “HELLO” TO YOUR NEW NEIGHBORS! Clooney Beg...
72118,1,LEFTY MEDIA DESPERATELY Tries To Bury Trump Bu...
72125,1,WOW! JILL STEIN’S ‘FIRESIDE CHAT’ Exposes Her ...
72128,1,JUDGE JEANINE SOUNDS FREE SPEECH ALARM: “They ...


In [16]:
df=df.drop_duplicates()

In [17]:
df.shape

(63121, 2)

In [18]:
## Check whether the dataset is an imbalance dataset or not
df['label'].value_counts()

label
0    34791
1    28330
Name: count, dtype: int64

In [19]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\panta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [21]:
stop_words = set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

In [22]:
def preprocess(text):
    ## Lower the text
    text = text.lower()
    ## remove html tags
    text = re.sub(r'<.*?>', '', text)
    ## remove urls
    text = re.sub(r'http\S+|www\S+', '', text)
    ## Remove special characters and numbers
    text = re.sub(r'[^a-z\s]+', '', text)
    ## Remove extra spaces 
    text = re.sub(r'\s+', ' ', text).strip()
    ## Remove stopwords and apply lemmatization
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text

In [23]:
df['Fake News'] = df['Fake News'].apply(preprocess)

In [24]:
df.head()

,label,Fake News
0,1,law enforcement high alert following threat co...
2,1,unbelievable obamas attorney general say charl...
3,0,bobby jindal raised hindu us story christian c...
4,1,satan russia unvelis image terrifying new supe...
5,1,time christian group sue amazon splc designati...


In [25]:
## Independent column
X=df['Fake News']
## Dependent Column
y=df['label']

In [26]:
## Apply Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [27]:
from nltk.tokenize import word_tokenize

In [28]:
from gensim.models import Word2Vec

## Train Data tokenization
X_train_tokens=X_train.apply(lambda x: word_tokenize(str(x)))

## Train word2vec model
word2vec_model=Word2Vec(sentences=X_train_tokens,vector_size=100,window=5,min_count=2,workers=4)

In [29]:
word2vec_model.wv['news']

array([ 0.7858402 , -1.6503167 ,  1.4334953 ,  1.0649931 ,  0.9885454 ,
        0.8186115 ,  0.3119899 , -0.05726264, -2.089058  ,  0.21621491,
       -1.9082801 , -0.4370788 , -0.43491927,  0.58200043, -0.5336705 ,
        0.17172804,  0.34610865,  2.7847593 ,  0.03243459,  0.45068163,
       -1.2932017 , -0.0600445 ,  0.23338011,  1.3766165 ,  1.7400395 ,
        5.8265266 ,  1.3808981 , -0.26117894, -0.44862208,  0.4228898 ,
       -0.9070262 ,  2.87704   ,  1.0141051 , -4.144748  , -0.9680688 ,
       -0.6611537 ,  0.6876443 , -0.89222956, -2.9026704 ,  0.14447589,
        0.8598951 ,  1.6138489 ,  1.0656804 ,  4.0896544 ,  3.8265858 ,
       -3.4482763 ,  5.0748506 ,  0.1496437 ,  0.77492124, -0.5795961 ,
       -0.8590041 ,  3.064525  ,  2.0150335 ,  2.1117604 , -4.1003075 ,
       -1.1837106 , -2.8282638 ,  2.5056348 ,  0.6739    ,  0.28826985,
        1.6048054 , -1.5978893 ,  1.9096391 ,  0.6902327 , -0.32428688,
       -1.1062514 , -2.1937766 , -4.1675773 , -0.5079559 ,  1.41

In [30]:
word2vec_model.wv.most_similar('news')

[('newsit', 0.7165990471839905),
 ('newsfacebook', 0.6847522258758545),
 ('newsthe', 0.6829640865325928),
 ('newswatch', 0.6764531135559082),
 ('newson', 0.6702380180358887),
 ('newshere', 0.6676635146141052),
 ('newsi', 0.6596888303756714),
 ('newshillaryclinton', 0.6586001515388489),
 ('presidentall', 0.6415702700614929),
 ('newsyou', 0.6333253979682922)]

In [31]:
X_test_tokens=X_test.apply(lambda x: word_tokenize(str(x)))

In [32]:
def get_sentence_vector(sentence):
    vectors=[]

    for word in sentence:
        if word in word2vec_model.wv:
            vectors.append(word2vec_model.wv[word])

    if(len(vectors))==0:
        return np.zeros(100)
    
    return np.mean(vectors,axis=0)

In [33]:
X_train_vectors=np.array([get_sentence_vector(sentence) for sentence in X_train_tokens])
X_test_vectors=np.array([get_sentence_vector(sentence) for sentence in X_test_tokens])

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
lr_model=LogisticRegression().fit(X_train_vectors,y_train)
svc_model=LinearSVC().fit(X_train_vectors,y_train)
gbc_model=GradientBoostingClassifier().fit(X_train_vectors,y_train)
rfc_model=RandomForestClassifier().fit(X_train_vectors,y_train)

In [35]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_lr=lr_model.predict(X_test_vectors)
y_pred_svc=svc_model.predict(X_test_vectors)
y_pred_gbc=gbc_model.predict(X_test_vectors)
y_pred_rfc=rfc_model.predict(X_test_vectors)

In [36]:
print("Accuracy Score of Logistic Regression is: ", accuracy_score(y_test,y_pred_lr))
print("Accuracy Score of Linear Support Vector Classifier is: ", accuracy_score(y_test,y_pred_svc))
print("Accuracy Score of Gradient Boosting Classifier is: ", accuracy_score(y_test,y_pred_gbc))
print("Accuracy Score of Random Forest Classifier is: ", accuracy_score(y_test,y_pred_rfc))

Accuracy Score of Logistic Regression is:  0.8955247524752475
Accuracy Score of Linear Support Vector Classifier is:  0.8940990099009901
Accuracy Score of Gradient Boosting Classifier is:  0.8780990099009901
Accuracy Score of Random Forest Classifier is:  0.8892673267326733


In [37]:
print("Classification Report of Logistic Regression is: \n",classification_report(y_test,y_pred_lr))
print("\n ======================================================== \n")
print("Classification Report of Linear Support Vector Classifier is: \n",classification_report(y_test,y_pred_svc))
print("\n ======================================================== \n")
print("Classification Report of Gradient Boosting Classifier is: \n",classification_report(y_test,y_pred_gbc))
print("\n ======================================================== \n")
print("Classification Report of Random Forest Classifier is: \n",classification_report(y_test,y_pred_rfc))

Classification Report of Logistic Regression is: 
               precision    recall  f1-score   support

           0       0.91      0.90      0.91      7038
           1       0.88      0.88      0.88      5587

    accuracy                           0.90     12625
   macro avg       0.89      0.89      0.89     12625
weighted avg       0.90      0.90      0.90     12625



Classification Report of Linear Support Vector Classifier is: 
               precision    recall  f1-score   support

           0       0.91      0.90      0.90      7038
           1       0.88      0.88      0.88      5587

    accuracy                           0.89     12625
   macro avg       0.89      0.89      0.89     12625
weighted avg       0.89      0.89      0.89     12625



Classification Report of Gradient Boosting Classifier is: 
               precision    recall  f1-score   support

           0       0.89      0.89      0.89      7038
           1       0.86      0.86      0.86      5587

   